## Imports

In [1]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

import EXP_neuro_fuzzy_toolbox as nft

In [2]:
import numpy as np

In [3]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Binary Classification

## Data

In [4]:
from ucimlrepo import fetch_ucirepo

In [5]:
parkinsons = fetch_ucirepo(id=174)

X = parkinsons.data.features
y = parkinsons.data.targets

In [6]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.5, stratify=y)

In [7]:
unique_classes, counts = np.unique(y_train.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1]
[24 73]


In [8]:
unique_classes, counts = np.unique(y_test.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1]
[24 74]


In [9]:
scaler = MinMaxScaler(feature_range=(0, 1))

x_train = scaler.fit_transform(x_train)

x_test = scaler.transform(x_test)

In [10]:
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

In [11]:
train_loader = data.DataLoader(
    data.TensorDataset(
        x_train, 
        y_train), 
    batch_size = 4, 
    shuffle = True)
x_train = train_loader.dataset.tensors[0]
y_train = train_loader.dataset.tensors[1]

## Model & Training

In [24]:
model = nft.rule_reduced_ANFIS(
    input_size = 22,
    num_mfs = 15,
    outputs = 2,
    output_type='softmax',
    dtype=torch.float64
)

model.init_premises(x_train)

In [25]:
loss_fn = nn.functional.cross_entropy

optimizer = torch.optim.AdamW
params = {'lr': 0.0001, 'weight_decay': 0.001}
#optimizer = torch.optim.Adam
#params = {'lr': 0.005}
#optimizer = torch.optim.SGD
#params = {'lr': 0.001, 'momentum': 0.9}

early_stopping = nft.EarlyStopping(
    patience=30, 
    delta=0.001
)

In [26]:
trainer = nft.Hybrid_learning_algorithm(
    epochs=500,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    validation=0.3,
    early_stopping=early_stopping
)

In [27]:
trainer(model, train_loader, verbose=True)

Epoch:   1/500 - loss: 0.704616 - validation loss: 0.723276
Epoch:   2/500 - loss: 0.500278 - validation loss: 0.484074
Epoch:   3/500 - loss: 0.703763 - validation loss: 0.720736
Epoch:   4/500 - loss: 0.699259 - validation loss: 0.716647
Epoch:   5/500 - loss: 0.413964 - validation loss: 0.629048
Epoch:   6/500 - loss: 0.697743 - validation loss: 0.708323
Epoch:   7/500 - loss: 0.698010 - validation loss: 0.706238
Epoch:   8/500 - loss: 0.698061 - validation loss: 0.705494
Epoch:   9/500 - loss: 0.611586 - validation loss: 0.687884
Epoch:  10/500 - loss: 0.698091 - validation loss: 0.705091
Epoch:  11/500 - loss: 0.698093 - validation loss: 0.704938
Epoch:  12/500 - loss: 0.698020 - validation loss: 0.703414
Epoch:  13/500 - loss: 0.697694 - validation loss: 0.700756
Epoch:  14/500 - loss: 0.697371 - validation loss: 0.699466
Epoch:  15/500 - loss: 0.693499 - validation loss: 0.688429
Epoch:  16/500 - loss: 0.608951 - validation loss: 0.618436
Epoch:  17/500 - loss: 0.692130 - valida

In [28]:
train_measures = nft.get_measures(model, x_train, y_train)
for measure in train_measures:
    print(measure + ':', train_measures[measure])

Accuracy: 0.8144329896907216
Precision: 0.8264498089688587
Recall: 0.8144329896907216
F1: 0.8189565625314137
Confusion Matrix: [[17  7]
 [11 62]]


In [29]:
test_measures = nft.get_measures(model, x_test, y_test)
for measure in test_measures:
    print(measure + ':', test_measures[measure])

Accuracy: 0.8367346938775511
Precision: 0.8326579435601991
Recall: 0.8367346938775511
F1: 0.8342738834664302
Confusion Matrix: [[15  9]
 [ 7 67]]
